In [2]:
# === Merge condition-level summaries from ./data/survay.csv and ./data/quantitative.csv ===
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("./data")
SURVAY_CSV = DATA_DIR / "survay.csv"
QUANT_CSV  = DATA_DIR / "quantitative.csv"
OUT_CSV    = DATA_DIR / "merged_condition_level.csv"

# --- 読み込み ---
sur = pd.read_csv(SURVAY_CSV)
qt  = pd.read_csv(QUANT_CSV)

# --- condition 正規化（すでに略称ならそのまま） ---
COND_ABBR = {
    "SpatialReference": "SR",
    "SpatialReference+Pointing": "P+SR",
    "Pointing": "Pointing",
    "Label": "Label",
    "空間参照だけ": "SR",
    "Pointing + 空間参照": "P+SR",
    "Pointing＋空間参照": "P+SR",
    "ラベル": "Label",
    # 同一表記はそのまま
    "SR": "SR", "P+SR": "P+SR", "Pointing": "Pointing", "Label": "Label",
}
def norm_cond(s):
    if pd.isna(s): return np.nan
    s = str(s).strip()
    return COND_ABBR.get(s, s)

# ========== 主観（survay.csv） ==========
# 期待カラム: section, table, scale, subscale, condition, median, mean, sd, n, ...
sur = sur.copy()
if "condition" not in sur.columns:
    raise KeyError("survay.csv に condition 列がありません（Descriptiveテーブルを想定）")
sur["condition"] = sur["condition"].map(norm_cond)

# 欲しいターゲット：SUS/TAM/TPA の Total、NASA-TLX の Overall
targets = [
    ("SUS",       "Total",   "SUS"),
    ("TAM",       "Total",   "TAM_Total"),
    ("TPA",       "Total",   "TPA_Total"),
    ("NASA-TLX",  "Overall", "NASA_Overall"),
]

def extract_sur_block(df, scale, subscale, out_prefix):
    sub = df[(df["section"]=="Descriptive") &
             (df["scale"]==scale) &
             (df["subscale"]==subscale)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["condition",
                                     f"{out_prefix}_median", f"{out_prefix}_mean",
                                     f"{out_prefix}_sd", f"{out_prefix}_n"])
    keep = ["condition","median","mean","sd","n"]
    for c in keep:
        if c not in sub.columns:
            sub[c] = np.nan
    sub = sub[keep]
    sub = sub.rename(columns={
        "median": f"{out_prefix}_median",
        "mean":   f"{out_prefix}_mean",
        "sd":     f"{out_prefix}_sd",
        "n":      f"{out_prefix}_n",
    })
    return sub

# 連結（conditionごとに横持ち）
sur_blocks = []
for (scale, subscale, prefix) in targets:
    sur_blocks.append(extract_sur_block(sur, scale, subscale, prefix))
sur_merged = None
for b in sur_blocks:
    sur_merged = b if sur_merged is None else sur_merged.merge(b, on="condition", how="outer")

# ========== 定量（quantitative.csv） ==========
# 期待カラム: section, table, metric, condition, median, mean, sd, ...
qt = qt.copy()
qt["condition"] = qt["condition"].map(norm_cond)

# metricごとに descriptive_by_condition を拾う
metrics_wanted = ["T_task_sec", "N_cmds", "N_actuations"]
sub_qt = qt[(qt["table"]=="descriptive_by_condition") & (qt["metric"].isin(metrics_wanted))].copy()

# 統計列が無いケースの埋め
for c in ["condition","metric","median","mean","sd"]:
    if c not in sub_qt.columns:
        sub_qt[c] = np.nan

# metric×condition を横展開（mean/median両方）
def wide_stat(df, stat_col, suffix):
    tmp = df.pivot_table(index="condition", columns="metric", values=stat_col, aggfunc="first")
    tmp = tmp.add_suffix(suffix).reset_index()
    return tmp

qt_med = wide_stat(sub_qt, "median", "_median")
qt_mean= wide_stat(sub_qt, "mean",   "_mean")
qt_sd  = wide_stat(sub_qt, "sd",     "_sd")

qt_wide = qt_med.merge(qt_mean, on="condition", how="outer").merge(qt_sd, on="condition", how="outer")

# logT を追加（mean/median 両方）
if "T_task_sec_mean" in qt_wide.columns:
    qt_wide["logT_mean"] = np.log(qt_wide["T_task_sec_mean"].clip(lower=1e-6))
if "T_task_sec_median" in qt_wide.columns:
    qt_wide["logT_median"] = np.log(qt_wide["T_task_sec_median"].clip(lower=1e-6))

# ========== 条件粒度のマージ ==========
if sur_merged is None or sur_merged.empty:
    raise RuntimeError("survay.csv から主観の Descriptive セクションが抽出できませんでした。")

merged = sur_merged.merge(qt_wide, on="condition", how="inner")

# 保存
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("✅ Merged (condition-level) saved:", OUT_CSV)
print("Rows/cols:", merged.shape)
display(merged)


✅ Merged (condition-level) saved: data\merged_condition_level.csv
Rows/cols: (4, 28)


,condition,SUS_median,SUS_mean,SUS_sd,SUS_n,TAM_Total_median,TAM_Total_mean,TAM_Total_sd,TAM_Total_n,TPA_Total_median,...,N_cmds_median,T_task_sec_median,N_actuations_mean,N_cmds_mean,T_task_sec_mean,N_actuations_sd,N_cmds_sd,T_task_sec_sd,logT_mean,logT_median
0,SR,52.50,50.937500,13.609709,24.0,4.291667,4.215278,1.035865,24.0,4.083333,...,2.0,63.105976,4.913043,2.065217,76.391011,2.489305,0.920775,35.350303,4.335865,4.144815
1,P+SR,83.75,80.520833,14.576991,24.0,6.000000,5.940972,0.840361,24.0,6.041667,...,1.0,37.329700,3.173913,1.152174,39.034887,1.275723,0.463062,14.263984,3.664456,3.619789
2,Pointing,76.25,78.333333,13.626753,24.0,5.500000,5.513889,0.855737,24.0,6.125000,...,2.0,34.545951,4.521739,2.021739,35.173758,1.016660,0.104257,9.855337,3.560300,3.542290
3,Label,42.50,47.916667,16.805581,24.0,3.833333,4.027778,1.302790,24.0,4.791667,...,1.5,51.547726,4.326087,1.673913,54.092099,1.336623,0.684383,17.160900,3.990688,3.942508


In [3]:
import pandas as pd

df = pd.read_csv("./data/merged_condition_level.csv")

# --- long（4〜5列）に整形 ---
# どの列を long 化するか（ファイルにある列名に合わせて調整）
value_cols = [c for c in df.columns if c != "condition"]

long4 = df.melt(id_vars=["condition"], value_vars=value_cols,
                var_name="metric", value_name="value")

# 例: metric名の末尾 _mean / _median を stat に分離したい場合（任意）
long4[["metric_base","stat"]] = long4["metric"].str.extract(r"^(.*)_(mean|median)$")
long4["metric_final"] = long4["metric_base"].fillna(long4["metric"])
long4 = long4.drop(columns=["metric_base"])
long4 = long4.rename(columns={"metric_final":"metric"})

# 保存
long4.to_csv("./data/merged_condition_level_long.csv", index=False, encoding="utf-8")
print("saved:", "./data/merged_condition_level_long.csv")

# --- wide（分析即戦力）も一緒にキープ（そのままが wide） ---
df.to_csv("./data/merged_condition_level_wide.csv", index=False, encoding="utf-8")
print("saved:", "./data/merged_condition_level_wide.csv")


saved: ./data/merged_condition_level_long.csv
saved: ./data/merged_condition_level_wide.csv


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
IMWUT-grade analysis (clean, single-file):
- Descriptives & plots (matplotlib only; no seaborn)
- Friedman + Wilcoxon (Holm)
- MixedLM for log(T_task_sec) with condition/devices/colors (+ interaction)
- GEE for N_cmds (family auto: Poisson or NegBin by overdispersion)
- Model-based pairwise contrasts for condition (Holm)
- Mediation hint (N_cmds as mediator for logT)
- Robustness check (top 1% trim)
- Within-subject Spearman (centering)
- *** NEW *** Subjective–Objective links:
    - Within/between Spearman (SUS/TAM/TPA/NASA vs logT, N_cmds)
    - MixedLM: Subjective ~ logT + N_cmds + C(condition)
    - Holm correction, markdown tables
- Markdown report to ./imwut_results/analysis_report.md

Inputs (must exist):
  ./imwut_results/task_summary_enriched.csv
  (optionally) ./imwut_results/participant_condition_summary.csv
  (optionally) ./imwut_results/SurvayResult.csv   # subjective per participant×condition

Additionally merges:
  ./RESULTS/P{participant}_{1..4}/*.json  (attempt array per file)
  -> sums metrics.tokens.system_total_time_sec per (participant, condition, taskId)
"""

import os
import re
import glob
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, wilcoxon, spearmanr, norm

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Poisson, NegativeBinomial
from statsmodels.genmod.cov_struct import Exchangeable

warnings.filterwarnings("ignore")

# ----------------- Constants & IO -----------------
OUT_DIR = "./imwut_results"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# 条件の順序とマップ
COND_ORDER = ["SR", "P+SR", "Pointing", "Label"]
COND_MAP   = {1: "SR", 2: "P+SR", 3: "Pointing", 4: "Label"}

TASKS_CSV = os.path.join(OUT_DIR, "task_summary_enriched.csv")
PC_CSV    = os.path.join(OUT_DIR, "participant_condition_summary.csv")
SURVEY_CSV= os.path.join(OUT_DIR, "SurvayResult.csv")  # 主観スコア（participant×condition）
REPORT_MD = os.path.join(OUT_DIR, "analysis_report.md")

RESULTS_ROOT = "./RESULTS"

# ----------------- Small utils -----------------
def is_json_file(path: str) -> bool:
    b = os.path.basename(path)
    return b.endswith(".json") and not b.endswith(".json.meta")

def safe_float(x, default=None):
    if x is None:
        return default
    try:
        if isinstance(x, (int, float)):
            return float(x)
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return default
        return float(s)
    except Exception:
        return default

def safe_get(d, *keys, default=None):
    cur = d
    try:
        for k in keys:
            if cur is None:
                return default
            cur = cur.get(k)
        return cur if cur is not None else default
    except Exception:
        return default

# ----------------- Helpers -----------------
def ensure_participant_column(df: pd.DataFrame) -> pd.DataFrame:
    """Find a participant-like column and normalize to 'participant'."""
    if "participant" in df.columns:
        return df
    for alt in ["participant_id", "subject", "user_id", "id", "P", "PID"]:
        if alt in df.columns:
            return df.rename(columns={alt: "participant"})
    raise KeyError("No participant column found. Expected one of: participant, participant_id, subject, user_id, id")

def normalize_condition_series(s: pd.Series) -> pd.Series:
    """多様な表記を SR / P+SR / Pointing / Label に揃える"""
    if s.dtype.kind in "iu":
        return s.map(COND_MAP).astype("category")
    m = {
        "SpatialReference": "SR",
        "SR": "SR",
        "Spatial Reference": "SR",
        "SpatialReference+Pointing": "P+SR",
        "Pointing+SpatialReference": "P+SR",
        "P+SR": "P+SR",
        "Pointing": "Pointing",
        "Label": "Label",
    }
    return s.astype(str).map(lambda x: m.get(x.strip(), x.strip())).astype("category")

def ecdf(arr):
    x = np.array([v for v in arr if pd.notna(v)])
    x.sort()
    if len(x) == 0:
        return np.array([]), np.array([])
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    out = np.zeros_like(pvals)
    prev = 0.0
    for k, idx in enumerate(order):
        adj = (m - k) * pvals[idx]
        adj = max(adj, prev)
        out[idx] = min(adj, 1.0)
        prev = out[idx]
    return out

def effect_r_from_W(W, n):
    mean_W = n * (n + 1) / 4.0
    var_W  = n * (n + 1) * (2 * n + 1) / 24.0
    if var_W <= 0:
        return 0.0
    z = (W - mean_W) / math.sqrt(var_W)
    return abs(z) / math.sqrt(n)

def winsorize_top(x, top=0.01):
    if len(x) == 0: return x
    q = np.quantile(x, 1-top)
    return np.clip(x, None, q)

def within_center(df: pd.DataFrame, value_col: str, by_col: str):
    """対象列をby_col単位でセンタリング"""
    return df[value_col] - df.groupby(by_col)[value_col].transform("mean")

def boxstrip(ax, values_by_group, labels, title, ylabel):
    ax.boxplot(values_by_group, labels=labels, showfliers=False)
    # strip (no color specification)
    for i, vals in enumerate(values_by_group, start=1):
        if vals is None or len(vals)==0: continue
        xj = np.random.normal(loc=i, scale=0.08, size=len(vals))
        ax.plot(xj, vals, 'o', alpha=0.6, markersize=3)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

# ----------------- RESULTS aggregator -----------------
def load_server_attempt_times_from_results(results_root: str):
    out_rows = []
    if not os.path.isdir(results_root):
        print(f"[WARN] RESULTS dir not found: {results_root}")
        return out_rows

    dirnames = [d for d in sorted(os.listdir(results_root)) if os.path.isdir(os.path.join(results_root, d))]
    pat = re.compile(r"^P(\d+)_([1-4])$")

    agg = {}  # key=(participant, condition, taskId) -> {sum, n}

    for dn in dirnames:
        m = pat.match(dn)
        if not m:
            continue
        participant_id = int(m.group(1))
        cond_num = int(m.group(2))
        condition = COND_MAP.get(cond_num)
        if not condition:
            continue

        dpath = os.path.join(results_root, dn)

        def _jsons_under(p):
            files = [fp for fp in glob.glob(os.path.join(p, "*.json")) if is_json_file(fp)]
            if not files:
                for sd in sorted(os.listdir(p)):
                    sp = os.path.join(p, sd)
                    if os.path.isdir(sp):
                        files.extend([fp for fp in glob.glob(os.path.join(sp, "*.json")) if is_json_file(fp)])
            return files

        for fp in _jsons_under(dpath):
            try:
                with open(fp, "r", encoding="utf-8") as f:
                    data = json.load(f)
            except Exception as e:
                print(f"[WARN] failed to read {fp}: {e}")
                continue

            # ← ← ← ここがエラーになっていた行：正しくは下の3枝分岐
            if isinstance(data, dict) and "results" in data and isinstance(data["results"], list):
                items = data["results"]
            elif isinstance(data, list):
                items = data
            else:
                continue

            for item in items:
                if not isinstance(item, dict):
                    continue
                task_id = item.get("task_id") or item.get("taskId")
                if not task_id:
                    continue

                sys_t = safe_get(item, "metrics", "tokens", "system_total_time_sec")
                if sys_t is None:
                    sys_t = safe_get(item, "metrics", "system_total_time_sec")
                sys_t = safe_float(sys_t, default=None)
                if sys_t is None:
                    continue

                key = (participant_id, condition, task_id)
                ag = agg.setdefault(key, {"sum": 0.0, "n": 0})
                ag["sum"] += sys_t
                ag["n"]   += 1

    for (pid, cond, tid), v in agg.items():
        out_rows.append({
            "participant": pid,
            "condition": cond,
            "taskId": tid,
            "system_time_total_sec": v["sum"],
            "n_attempts_server": v["n"],
        })
    return out_rows

# ----------------- Load & merge -----------------
def load_inputs():
    if not os.path.exists(TASKS_CSV):
        raise FileNotFoundError(f"Missing: {TASKS_CSV}")

    # 1) タスクCSV読み込み
    df_tasks = pd.read_csv(TASKS_CSV)
    df_tasks = ensure_participant_column(df_tasks)
    if "condition" not in df_tasks.columns:
        # 1..4 が入っている場合にも対応
        for cand in ["cond", "Condition", "condition_id"]:
            if cand in df_tasks.columns:
                df_tasks = df_tasks.rename(columns={cand:"condition"})
                break
    df_tasks["condition"] = normalize_condition_series(df_tasks["condition"]).cat.set_categories(COND_ORDER, ordered=True)

    if "n_devices_required" not in df_tasks.columns and "n_devices_final" in df_tasks.columns:
        df_tasks["n_devices_required"] = df_tasks["n_devices_final"]

    # 2) RESULTS から system_total_time_sec を集計してマージ
    server_rows = load_server_attempt_times_from_results(RESULTS_ROOT)
    if server_rows:
        df_server = pd.DataFrame(server_rows)
        df_tasks = df_tasks.merge(df_server, on=["participant","condition","taskId"], how="left")
    else:
        print("[WARN] No server attempt time rows merged.")

    # 3) 参加者×条件メディアン（df_pc）
    agg_cols = [c for c in [
        "T_task_sec","N_cmds","N_actuations",
        "n_devices_final","n_devices_required","n_colors_required",
        "system_time_total_sec","n_attempts_server"
    ] if c in df_tasks.columns]

    df_pc = (df_tasks
             .groupby(["participant","condition"], as_index=False)[agg_cols]
             .median(numeric_only=True))

    return df_tasks, df_pc

# ----------------- Subjective loader (NEW) -----------------
def load_subjective_pc():
    """
    SurvayResult.csv を participant×condition に揃えて返す。
    想定列: participant / condition / (SUS*, TAM*, TPA*, NASA*)
    大小文字無視・部分一致で代表列を自動選択（Total/合計があれば優先）
    """
    if not os.path.exists(SURVEY_CSV):
        print(f"[INFO] Subjective CSV not found: {SURVEY_CSV}")
        return None

    df = pd.read_csv(SURVEY_CSV)
    df = ensure_participant_column(df)

    # condition 列を用意
    cond_col = None
    for c in ["condition","Condition","cond","Cond"]:
        if c in df.columns:
            cond_col = c; break
    if cond_col is None:
        # 1..4 の列（例: condition_id）でも良い
        for c in ["condition_id","cond_id","C"]:
            if c in df.columns:
                cond_col = c; break
    if cond_col is None:
        raise KeyError("Subjective CSV needs a condition column.")

    df = df.rename(columns={cond_col:"condition"})
    df["condition"] = normalize_condition_series(df["condition"]).astype("category")

    # 代表列を柔軟に拾う
    def pick_one(prefixes):
        # 優先: Total / sum / overall を含む列
        cols = [c for c in df.columns if any(p.lower() in c.lower() for p in prefixes)]
        if not cols:
            return None
        # Total優先
        total_like = [c for c in cols if any(k in c.lower() for k in ["total","sum","overall","score"])]
        return total_like[0] if total_like else cols[0]

    sus_col  = pick_one(["SUS"])
    tam_col  = pick_one(["TAM"])
    tpa_col  = pick_one(["TPA","trust"])
    nasa_col = pick_one(["NASA","TLX","NASA-TLX"])

    keep = ["participant","condition"] + [c for c in [sus_col,tam_col,tpa_col,nasa_col] if c]
    df = df[keep].copy()

    rename = {}
    if sus_col:  rename[sus_col]  = "SUS"
    if tam_col:  rename[tam_col]  = "TAM"
    if tpa_col:  rename[tpa_col]  = "TPA"
    if nasa_col: rename[nasa_col] = "NASA_TLX"

    df = df.rename(columns=rename)

    # 数値化（失敗したら NaN）
    for c in ["SUS","TAM","TPA","NASA_TLX"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

# ----------------- Descriptive Plots -----------------
def plot_descriptive(df_pc: pd.DataFrame):
    metrics = [m for m in ["T_task_sec", "N_cmds", "N_actuations", "system_time_total_sec"] if m in df_pc.columns]
    for metric in metrics:
        vals = [df_pc[df_pc["condition"]==c][metric].dropna().values for c in COND_ORDER]
        fig, ax = plt.subplots(figsize=(6.5,4.2))
        boxstrip(ax, vals, COND_ORDER, f"{metric} by Condition", metric)
        fig.tight_layout()
        fig.savefig(os.path.join(PLOT_DIR, f"boxstrip_{metric}.png"), dpi=220)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(6.5,4.2))
        for c in COND_ORDER:
            arr = df_pc[df_pc["condition"]==c][metric].dropna().values
            x,y = ecdf(arr)
            if len(x)==0: continue
            ax.plot(x,y,label=c)
        ax.set_title(f"ECDF of {metric} by Condition")
        ax.set_xlabel(metric); ax.set_ylabel("F(x)")
        ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(PLOT_DIR, f"ecdf_{metric}.png"), dpi=220)
        plt.close(fig)

# ----------------- Friedman + Wilcoxon -----------------
def friedman_block(df_pc: pd.DataFrame, metric: str):
    if metric not in df_pc.columns:
        return None, None, None, None
    pivot = df_pc.pivot(index="participant", columns="condition", values=metric)
    pivot = pivot.reindex(columns=COND_ORDER).dropna(how="any")
    if len(pivot)==0:
        return None, None, None, None
    arrays = [pivot[c].values for c in COND_ORDER]
    stat, p = friedmanchisquare(*arrays)

    from itertools import combinations
    pairs, Ws, pvals, ns, rs = [], [], [], [], []
    for a,b in combinations(COND_ORDER, 2):
        x = pivot[a].values; y = pivot[b].values
        n = len(x)
        try:
            W, pw = wilcoxon(x, y, zero_method='wilcox', correction=False)
        except Exception:
            W, pw = (0.0, 1.0)
        r = effect_r_from_W(W, n)
        pairs.append(f"{a} vs {b}")
        Ws.append(W)
        pvals.append(pw)
        ns.append(n)
        rs.append(r)
    p_holm = holm_adjust(pvals).tolist()
    table = pd.DataFrame({"pair":pairs, "n":ns, "W":Ws, "p_raw":pvals, "p_holm":p_holm, "effect_r":rs})
    return stat, p, pivot, table

# ----------------- Modeling -----------------
def prepare_model_frame(df_tasks: pd.DataFrame) -> pd.DataFrame:
    dfm = df_tasks.copy()
    required = ["participant","condition","T_task_sec","N_cmds","n_devices_required"]
    missing = [c for c in required if c not in dfm.columns]
    if missing:
        raise KeyError(f"Missing columns in task CSV: {missing}")

    dfm["logT"] = np.log(dfm["T_task_sec"].clip(lower=1e-6))

    for col in ["n_devices_required","n_colors_required"]:
        if col not in dfm.columns:
            dfm[col] = np.nan
        mu = dfm[col].dropna().mean() if dfm[col].notna().any() else 0.0
        dfm[col+"_c"] = dfm[col] - mu
    return dfm

def fit_mixedlm_logT(dfm: pd.DataFrame):
    sub = dfm.dropna(subset=["logT","n_devices_required_c","n_colors_required_c"])
    if sub.empty:
        return None
    try:
        formula = "logT ~ C(condition) + n_devices_required_c + n_colors_required_c + n_devices_required_c:C(condition)"
        m = smf.mixedlm(formula, data=sub, groups=sub["participant"])
        return m.fit(method="lbfgs", reml=False)
    except Exception as e:
        print("[WARN] MixedLM(logT) failed:", e)
        return None

def summarize_mixedlm_logT(res) -> str:
    if res is None:
        return "MixedLM(logT) did not converge or no data.\n"
    lines = []
    lines.append("### MixedLM: log(T_task_sec) ~ condition + devices + colors + condition×devices\n")
    lines.append("Fixed effects (β), 95%CI, and % change (exp(β)-1)*100.\n")
    params = res.params
    conf = res.conf_int()
    rows = []
    for name, beta in params.items():
        ci = conf.loc[name].values if name in conf.index else [np.nan, np.nan]
        pct = (math.exp(beta)-1)*100
        rows.append([name, beta, ci[0], ci[1], pct])
    out = pd.DataFrame(rows, columns=["term","beta","ci_low","ci_high","pct_change"])
    lines.append(out.to_string(index=False))
    lines.append("\n")
    return "\n".join(lines)

def choose_family_by_overdispersion(dfm: pd.DataFrame):
    y = dfm["N_cmds"].dropna().values
    if len(y)==0: return Poisson()
    mu = np.mean(y)
    va = np.var(y, ddof=1) if len(y)>1 else mu
    return NegativeBinomial() if va > mu*1.2 else Poisson()

def fit_gee_ncmds(dfm: pd.DataFrame):
    sub = dfm.dropna(subset=["N_cmds","n_devices_required_c","n_colors_required_c"])
    if sub.empty:
        return None
    try:
        family = choose_family_by_overdispersion(sub)
        formula = "N_cmds ~ C(condition) + n_devices_required_c + n_colors_required_c + n_devices_required_c:C(condition)"
        model = GEE.from_formula(formula, groups="participant", data=sub, cov_struct=Exchangeable(), family=family)
        return model.fit()
    except Exception as e:
        print("[WARN] GEE failed:", e)
        return None

def summarize_gee(res) -> str:
    if res is None:
        return "GEE(N_cmds) failed or no data.\n"
    lines = []
    lines.append(f"### GEE: N_cmds ~ condition + devices + colors + condition×devices  (family={res.model.family.__class__.__name__})\n")
    lines.append("Coefficients as IRR=exp(β), 95%CI.\n")
    params = res.params
    conf = res.conf_int()
    rows = []
    for name, beta in params.items():
        ci = conf.loc[name].values if name in conf.index else [np.nan, np.nan]
        irr, lo, hi = math.exp(beta), math.exp(ci[0]), math.exp(ci[1])
        rows.append([name, beta, ci[0], ci[1], irr, lo, hi])
    out = pd.DataFrame(rows, columns=["term","beta","ci_low","ci_high","IRR","IRR_low","IRR_high"])
    lines.append(out.to_string(index=False))
    lines.append("\n")
    return "\n".join(lines)

def pairwise_condition_contrasts_gee(res):
    if res is None:
        return None
    names = res.model.exog_names

    def coef_for_cond(level):
        v = np.zeros(len(names))
        if "Intercept" in names:
            v[names.index("Intercept")] = 1.0
        for n in names:
            if n.startswith("C(condition)") and f"[T.{level}]" in n:
                v[names.index(n)] = 1.0
        return v

    rows = []
    from itertools import combinations
    for a, b in combinations(COND_ORDER, 2):
        va = coef_for_cond(a); vb = coef_for_cond(b)
        diff = va - vb
        est = float(diff @ res.params.values)
        V = res.cov_params()
        se = math.sqrt(float(diff.T @ V.values @ diff))
        z  = (est / se) if se > 0 else 0.0
        p  = 2 * (1 - norm.cdf(abs(z)))
        irr_ratio = math.exp(est)
        rows.append([f"{a} vs {b}", est, se, z, p, irr_ratio])
    dfc = pd.DataFrame(rows, columns=["pair","est","se","z","p_raw","IRR_ratio"])
    dfc["p_holm"] = holm_adjust(dfc["p_raw"].values)
    return dfc

def mediation_hint(dfm: pd.DataFrame) -> str:
    sub = dfm[["participant","condition","logT","N_cmds","n_devices_required_c","n_colors_required_c"]].dropna()
    if sub.empty:
        return "Mediation: insufficient data.\n"
    try:
        m_red  = smf.mixedlm("logT ~ C(condition) + n_devices_required_c + n_colors_required_c", sub, groups=sub["participant"]).fit(method="lbfgs", reml=False)
        m_full = smf.mixedlm("logT ~ C(condition) + n_devices_required_c + n_colors_required_c + N_cmds", sub, groups=sub["participant"]).fit(method="lbfgs", reml=False)
        def coef_table(res, tag):
            coefs = res.params.filter(like="C(condition)")
            return pd.DataFrame({"term":coefs.index, tag:coefs.values})
        t = coef_table(m_red,"beta_reduced").merge(coef_table(m_full,"beta_full"), on="term", how="outer")
        t["shrink"] = (t["beta_reduced"].abs() - t["beta_full"].abs()) / t["beta_reduced"].abs()
        return "### Mediation (heuristic): effect shrinkage when adding N_cmds\n" + t.to_string(index=False) + "\n"
    except Exception as e:
        return f"Mediation failed: {e}\n"

def robustness_trim_logT(dfm: pd.DataFrame):
    sub = dfm.copy()
    sub["T_trim"] = winsorize_top(sub["T_task_sec"].values, top=0.01)
    sub["logT_trim"] = np.log(sub["T_trim"].clip(lower=1e-6))
    sub = sub.dropna(subset=["logT_trim","n_devices_required_c","n_colors_required_c"])
    if sub.empty:
        return None
    try:
        res = smf.mixedlm("logT_trim ~ C(condition) + n_devices_required_c + n_colors_required_c + n_devices_required_c:C(condition)",
                          sub, groups=sub["participant"]).fit(method="lbfgs", reml=False)
        return res
    except Exception:
        return None

def within_subject_spearman(df_tasks: pd.DataFrame) -> str:
    sub = ensure_participant_column(df_tasks.copy())

    sub["logT"] = np.log(sub["T_task_sec"].clip(lower=1e-6))
    if "n_devices_required" not in sub.columns and "n_devices_final" in sub.columns:
        sub["n_devices_required"] = sub["n_devices_final"]

    sub["logT_w"] = within_center(sub, "logT", "participant")
    sub["dev_w"]  = within_center(sub, "n_devices_required", "participant")
    if "n_colors_required" in sub.columns:
        sub["col_w"] = within_center(sub, "n_colors_required", "participant")
    else:
        sub["col_w"] = np.nan

    out = []
    for y, x in [("logT_w", "dev_w"), ("logT_w", "col_w")]:
        s = sub[[y, x]].dropna()
        if len(s) >= 5:
            rho, p = spearmanr(s[y], s[x])
            out.append(f"- within Spearman ρ({y}, {x}) = {rho:.3f}, p={p:.3g}, N={len(s)}")
        else:
            out.append(f"- within Spearman ρ({y}, {x}) = n/a (N<{len(s)})")
    return "\n".join(out) + "\n"

# ----------------- Subjective–Objective links (NEW) -----------------
def subjective_objective_links(df_pc: pd.DataFrame, df_sj: pd.DataFrame) -> str:
    """
    参加者×条件単位で主観(SUS/TAM/TPA/NASA_TLX)と客観(T_task_sec, N_cmds)の関連を評価。
    - Spearman（between / within）
    - MixedLM: Subjective ~ logT + N_cmds + C(condition)
    """
    lines = []
    if df_sj is None:
        return "## 主観×客観の関連\n主観データが見つからないため、この解析はスキップしました。\n"

    # マージ（participant×condition）
    use_cols_pc = ["participant","condition"] + [c for c in ["T_task_sec","N_cmds"] if c in df_pc.columns]
    m = pd.merge(df_pc[use_cols_pc], df_sj, on=["participant","condition"], how="inner")
    if m.empty:
        return "## 主観×客観の関連\n主観と客観の結合データが空でした。\n"

    # logT（メディアンベース）
    if "T_task_sec" in m.columns:
        m["logT"] = np.log(m["T_task_sec"].clip(lower=1e-6))

    subj_cols = [c for c in ["SUS","TAM","TPA","NASA_TLX"] if c in m.columns]
    obj_cols  = [c for c in ["logT","N_cmds"] if c in m.columns]

    if not subj_cols or not obj_cols:
        return "## 主観×客観の関連\n解析に必要な列（SUS/TAM/TPA/NASA_TLX と logT/N_cmds）が不足しています。\n"

    # スピアマン（全体）
    rows = []
    pvals = []
    for s in subj_cols:
        for o in obj_cols:
            d = m[[s, o]].dropna()
            if len(d) >= 6:
                rho, p = spearmanr(d[s], d[o])
                rows.append([s, o, len(d), rho, p])
                pvals.append(p)
    if rows:
        adj = holm_adjust(pvals)
        for i, a in enumerate(adj):
            rows[i].append(float(a))
    lines.append("## 主観×客観の関連\n")
    if rows:
        df_corr = pd.DataFrame(rows, columns=["subjective","objective","N","rho","p_raw","p_holm"])
        lines.append("### Spearman（全体）\n" + df_corr.to_string(index=False) + "\n")
    else:
        lines.append("### Spearman（全体）\n有効なペアがありませんでした。\n")

    # 被験者内センタリング Spearman
    wrows, wpvals = [], []
    # センタリングは participant×condition の表（m）に対して participant 内で
    for s in subj_cols:
        m[f"{s}_w"] = within_center(m, s, "participant")
    if "logT" in m.columns:
        m["logT_w"] = within_center(m, "logT", "participant")
    if "N_cmds" in m.columns:
        m["N_cmds_w"] = within_center(m, "N_cmds", "participant")

    wpairs = []
    if "logT_w" in m.columns: wpairs.append(("logT_w","時間（logT）"))
    if "N_cmds_w" in m.columns: wpairs.append(("N_cmds_w","コマンド回数（N_cmds）"))

    for s in subj_cols:
        for o_col, o_label in wpairs:
            d = m[[f"{s}_w", o_col]].dropna()
            if len(d) >= 6:
                rho, p = spearmanr(d[f"{s}_w"], d[o_col])
                wrows.append([s, o_label, len(d), rho, p])
                wpvals.append(p)
    if wrows:
        adj = holm_adjust(wpvals)
        for i, a in enumerate(adj):
            wrows[i].append(float(a))
        df_wc = pd.DataFrame(wrows, columns=["subjective","objective(centered)","N","rho","p_raw","p_holm"])
        lines.append("### 被験者内 Spearman（センタリング）\n" + df_wc.to_string(index=False) + "\n")
    else:
        lines.append("### 被験者内 Spearman（センタリング）\n有効なペアがありませんでした。\n")

    # MixedLM: Subjective ~ logT + N_cmds + C(condition)
    for s in subj_cols:
        sub = m.dropna(subset=[s] + obj_cols)
        if sub.empty:
            lines.append(f"### MixedLM（{s} ~ logT + N_cmds + C(condition)）\nデータ不足\n")
            continue
        try:
            # ある方がない場合に耐性
            rhs = []
            if "logT" in sub.columns: rhs.append("logT")
            if "N_cmds" in sub.columns: rhs.append("N_cmds")
            rhs.append("C(condition)")
            formula = f"{s} ~ " + " + ".join(rhs)
            res = smf.mixedlm(formula, data=sub, groups=sub["participant"]).fit(method="lbfgs", reml=False)
            params = res.params
            conf   = res.conf_int()
            rows = []
            for nm, beta in params.items():
                ci = conf.loc[nm].values if nm in conf.index else [np.nan, np.nan]
                rows.append([nm, beta, ci[0], ci[1]])
            tdf = pd.DataFrame(rows, columns=["term","beta","ci_low","ci_high"])
            lines.append(f"### MixedLM（{s} ~ logT + N_cmds + C(condition)）\n" + tdf.to_string(index=False) + "\n")
        except Exception as e:
            lines.append(f"### MixedLM（{s}）失敗: {e}\n")

    return "\n".join(lines)

# ----------------- Report blocks -----------------
def write_friedman_blocks(lines, df_pc: pd.DataFrame):
    metrics = [m for m in ["T_task_sec","N_cmds","N_actuations","system_time_total_sec"] if m in df_pc.columns]
    for metric in metrics:
        stat, p, pivot, pairs = friedman_block(df_pc, metric)
        lines.append(f"\n## {metric}\n")
        if stat is None:
            lines.append("有効データ不足\n"); continue
        lines.append(f"\nFriedman: χ²={stat:.4f}, p={p:.6g}, N={len(pivot)}\n")
        lines.append("\n### 記述統計（参加者×条件メディアン）\n")
        lines.append("Condition | Median | Q1 | Q3 | Mean | SD")
        lines.append("---|---:|---:|---:|---:|---:")
        for c in COND_ORDER:
            vals = pivot[c].dropna().values if c in pivot.columns else np.array([])
            if len(vals)==0:
                lines.append(f"{c} |  |  |  |  | "); continue
            med = np.median(vals); q1=np.percentile(vals,25); q3=np.percentile(vals,75)
            mu = np.mean(vals); sd = np.std(vals, ddof=1) if len(vals)>1 else 0.0
            lines.append(f"{c} | {med:.3g} | {q1:.3g} | {q3:.3g} | {mu:.3g} | {sd:.3g}")
        if p < 0.05 and pairs is not None:
            lines.append("\n### ペアごとの Wilcoxon（Holm 補正）\n")
            lines.append("Pair | n | W | p_raw | p_holm | effect_r")
            lines.append("---|---:|---:|---:|---:|---:")
            for _, r in pairs.iterrows():
                lines.append(f"{r['pair']} | {int(r['n'])} | {r['W']:.3g} | {r['p_raw']:.3g} | {r['p_holm']:.3g} | {r['effect_r']:.3g}")
        else:
            lines.append("\n（Friedman で有意差なし → ペア検定省略）\n")

def write_models_blocks(lines, dfm: pd.DataFrame):
    mixed_logT = fit_mixedlm_logT(dfm)
    lines.append("\n## 混合効果モデル：log(T_task_sec)\n")
    lines.append(summarize_mixedlm_logT(mixed_logT))

    gee_res = fit_gee_ncmds(dfm)
    lines.append("\n## GEE：N_cmds（カウント；過分散に応じ Poisson/NB）\n")
    lines.append(summarize_gee(gee_res))

    gee_pairs = pairwise_condition_contrasts_gee(gee_res)
    if gee_pairs is not None and len(gee_pairs):
        lines.append("\n### 条件のモデルベース比較（IRR比；Holm補正）\n")
        lines.append("Pair | est | se | z | p_raw | p_holm | IRR_ratio")
        lines.append("---|---:|---:|---:|---:|---:|---:")
        for _, r in gee_pairs.iterrows():
            lines.append(
                f"{r['pair']} | {r['est']:.3g} | {r['se']:.3g} | {r['z']:.3g} | {r['p_raw']:.3g} | {r['p_holm']:.3g} | {r['IRR_ratio']:.3g}"
            )

    lines.append("\n## 媒介の示唆（N_cmds が logT を媒介するか）\n")
    lines.append(mediation_hint(dfm))

    lines.append("\n## ロバスト性（上位1%トリムの再推定）\n")
    rob = robustness_trim_logT(dfm)
    lines.append(summarize_mixedlm_logT(rob))

    lines.append("\n## 色×条件 交互作用（logT, MixedLM）\n")

    def fit_mixedlm_logT_with_color_interaction(dfm: pd.DataFrame):
        sub = dfm.dropna(subset=["logT","n_devices_required_c","n_colors_required_c"])
        if sub.empty:
            return None
        formula = (
            "logT ~ C(condition)"
            " + n_devices_required_c"
            " + n_colors_required_c"
            " + n_devices_required_c:C(condition)"
            " + n_colors_required_c:C(condition)"
        )
        try:
            m = smf.mixedlm(formula, data=sub, groups=sub["participant"])
            return m.fit(method="lbfgs", reml=False)
        except Exception as e:
            print("[WARN] MixedLM(logT, color x condition) failed:", e)
            return None

    mix_color = fit_mixedlm_logT_with_color_interaction(dfm)
    if mix_color is None:
        lines.append("色×条件の交互作用モデルが推定できませんでした。\n")
        return

    params_fe = mix_color.fe_params
    conf_fe   = mix_color.conf_int().loc[params_fe.index]
    tmp = []
    for nm, beta in params_fe.items():
        ci_low, ci_high = conf_fe.loc[nm].values
        pct = (math.exp(beta)-1)*100
        tmp.append([nm, beta, ci_low, ci_high, pct])
    df_out = pd.DataFrame(tmp, columns=["term","beta","ci_low","ci_high","pct_change"])
    lines.append(df_out.to_string(index=False) + "\n")

    lines.append("\n### 色ごとの P+SR vs Pointing の予測時間比（<1: P+SRが速い）\n")

    def contrasts_PSR_vs_Pointing_by_colors(mix_res, dfm):
        names = mix_res.model.exog_names
        beta  = mix_res.fe_params.loc[names].values
        Vfull = mix_res.cov_params()
        V     = Vfull.loc[names, names].values

        col_vals = sorted(dfm["n_colors_required"].dropna().unique())
        rows = []

        def vec(cond, col_raw):
            col_c = col_raw - dfm["n_colors_required"].dropna().mean()
            x = np.zeros(len(names))
            if "Intercept" in names:
                x[names.index("Intercept")] = 1.0
            for n in names:
                if n.startswith("C(condition)") and f"[T.{cond}]" in n:
                    x[names.index(n)] = 1.0
            if "n_colors_required_c" in names:
                x[names.index("n_colors_required_c")] = col_c
            key = f"n_colors_required_c:C(condition)[T.{cond}]"
            if key in names:
                x[names.index(key)] = col_c
            return x

        for col_raw in col_vals:
            xa = vec("P+SR", col_raw)
            xb = vec("Pointing", col_raw)
            diff = xa - xb
            est = float(diff @ beta)
            se  = float(np.sqrt(diff @ V @ diff))
            z   = est / se if se > 0 else 0.0
            p   = 2 * (1 - norm.cdf(abs(z)))
            ratio = math.exp(est)
            rows.append([col_raw, est, se, z, p, ratio])

        out = pd.DataFrame(rows, columns=["colors","est_logT","se","z","p_raw","time_ratio_PSR/Pointing"])
        out["p_holm"] = holm_adjust(out["p_raw"].values)
        return out

    ct = contrasts_PSR_vs_Pointing_by_colors(mix_color, dfm)
    lines.append(ct.to_string(index=False) + "\n")

def write_within_corr_and_QA(lines, df_tasks: pd.DataFrame):
    lines.append("\n## 被験者内スピアマン相関（センタリング）\n")
    lines.append(within_subject_spearman(df_tasks))

    colors_unique  = sorted(df_tasks["n_colors_required"].dropna().unique().tolist()) if "n_colors_required" in df_tasks.columns else []
    devices_unique = sorted(df_tasks["n_devices_required"].dropna().unique().tolist()) if "n_devices_required" in df_tasks.columns else []
    lines.append("\n## 変動の確認\n")
    lines.append(f"- devices unique values: {devices_unique}\n")
    lines.append(f"- colors unique values:  {colors_unique}\n")

# ----------------- Optional stub -----------------
def plot_partial_effect(dfm: pd.DataFrame):
    return

# ----------------- Main -----------------
def main():
    # 1) データ読み込み & RESULTS マージ
    df_tasks, df_pc = load_inputs()

    # 2) 記述統計プロット
    plot_descriptive(df_pc)

    # 3) モデル用データ整形
    dfm = prepare_model_frame(df_tasks)

    # 4) 主観データ読み込み（NEW）
    df_sj = None
    try:
        df_sj = load_subjective_pc()
    except Exception as e:
        print(f"[WARN] Failed to load subjective CSV: {e}")

    # 5) Markdown レポート
    lines = ["# 高度統計レポート（IMWUT向け）\n"]

    write_friedman_blocks(lines, df_pc)
    write_models_blocks(lines, dfm)
    write_within_corr_and_QA(lines, df_tasks)

    # 6) 主観×客観の関連（NEW）
    lines.append("\n")
    lines.append(subjective_objective_links(df_pc, df_sj))

    # 7) ファイル出力
    with open(REPORT_MD, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[OK] wrote {REPORT_MD}")

    # 8) Optional
    try:
        plot_partial_effect(dfm)
    except Exception:
        pass
main()

IndentationError: unexpected indent (190390768.py, line 716)